[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-4/parallelization.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239934-lesson-1-parallelization)

# 并行节点执行

## 回顾

在模块 3 中，我们深入探讨了`人工干预`，展示了 3 个常见用例：

(1) `批准` - 我们可以中断智能体，向用户显示状态，并允许用户接受某个操作

(2) `调试` - 我们可以回退图以重现或避免问题

(3) `编辑` - 您可以修改状态

## 目标

本模块将基于`人工干预`以及模块 2 中讨论的`记忆`概念进行构建。

我们将深入探讨`多智能体`工作流，并构建一个多智能体研究助理，它将本课程所有模块的内容整合在一起。

为了构建这个多智能体研究助理，我们首先将讨论一些 LangGraph 可控性主题。

我们将从[并行化](https://langchain-ai.github.io/langgraph/how-tos/branching/#how-to-create-branches-for-parallel-node-execution)开始。

## 扇出和扇入

让我们构建一个简单的线性图，在每一步都重写状态。

In [ ]:
%%capture --no-stderr
%pip install -U  langgraph tavily-python wikipedia langchain_openai langchain_community langgraph_sdk

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [ ]:
from IPython.display import Image, display

from typing import Any
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    # operator.add reducer 函数使其只能追加
    state: str

class ReturnNodeValue:
    def __init__(self, node_secret: str):
        self._value = node_secret

    def __call__(self, state: State) -> Any:
        print(f"Adding {self._value} to {state['state']}")
        return {"state": [self._value]}

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", "c")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

正如预期的那样，我们重写了状态。

In [ ]:
graph.invoke({"state": []})

现在，让我们并行运行 `b` 和 `c`。

然后运行 `d`。

我们可以通过从 `a` 扇出到 `b` 和 `c`，然后扇入到 `d` 来轻松做到这一点。

状态更新在每一步的结尾应用。

让我们运行它。

In [ ]:
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "d")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

**我们看到一个错误**！

这是因为 `b` 和 `c` 都在同一步写入相同的状态键/通道。

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    graph.invoke({"state": []})
except InvalidUpdateError as e:
    print(f"An error occurred: {e}")

使用扇出时，如果步骤正在写入相同的通道/键，我们需要确保使用 reducer。

正如我们在模块 2 中提到的，`operator.add` 是 Python 内置 operator 模块中的一个函数。

当 `operator.add` 应用于列表时，它执行列表连接。

In [ ]:
import operator
from typing import Annotated

class State(TypedDict):
    # operator.add reducer 函数使其只能追加
    state: Annotated[list, operator.add]

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "d")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"state": []})

现在我们看到，我们为 `b` 和 `c` 并行进行的更新追加到状态中。

## 等待节点完成

现在，让我们考虑一种情况，其中一个并行路径的步骤比另一个多。

In [ ]:
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("b2", ReturnNodeValue("I'm B2"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "b2")
builder.add_edge(["b2", "c"], "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

在这种情况下，`b`、`b2` 和 `c` 都是同一步的一部分。

图将等待所有这些完成，然后才进行到步骤 `d`。

In [ ]:
graph.invoke({"state": []})

## 设置状态更新顺序

然而，在每一步中，我们对状态更新的顺序没有特定的控制！

简单来说，这是由 LangGraph 基于图拓扑确定的确定性顺序，**我们无法控制**。

上面，我们看到 `c` 在 `b2` 之前被添加。

然而，我们可以使用自定义 reducer 来自定义这个过程，例如，对状态更新进行排序。

In [ ]:
def sorting_reducer(left, right):
    """ 组合和排序列表中的值"""
    if not isinstance(left, list):
        left = [left]

    if not isinstance(right, list):
        right = [right]
    
    return sorted(left + right, reverse=False)

class State(TypedDict):
    # sorting_reducer 将对状态中的值进行排序
    state: Annotated[list, sorting_reducer]

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("b2", ReturnNodeValue("I'm B2"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "b2")
builder.add_edge(["b2", "c"], "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"state": []})

现在，reducer 对更新的状态值进行排序！

`sorting_reducer` 示例对所有值进行全局排序。我们还可以：

1. 在并行步骤期间将输出写入状态中的单独字段
2. 在并行步骤后使用"sink"节点来合并和排序这些输出
3. 合并后清除临时字段

有关更多详细信息，请参阅[文档](https://langchain-ai.github.io/langgraph/how-tos/branching/#stable-sorting)。

## 使用 LLM

现在，让我们添加一个实际的例子！

我们想要从两个外部源（Wikipedia 和 Web 搜索）收集上下文，并让 LLM 回答问题。

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0) 

In [ ]:
class State(TypedDict):
    question: str
    answer: str
    context: Annotated[list, operator.add]

您可以尝试不同的网络搜索工具。[Tavily](https://tavily.com/) 是一个不错的选择，但请确保设置了您的 `TAVILY_API_KEY`。

In [ ]:
import os, getpass
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("TAVILY_API_KEY")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

from langchain_community.document_loaders import WikipediaLoader
from langchain_community.tools import TavilySearchResults

def search_web(state):
    
    """ 从网络搜索检索文档 """

    # 搜索
    tavily_search = TavilySearchResults(max_results=3)
    search_docs = tavily_search.invoke(state['question'])

     # 格式化
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document href="{doc["url"]}">\n{doc["content"]}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]} 

def search_wikipedia(state):
    
    """ 从 wikipedia 检索文档 """

    # 搜索
    search_docs = WikipediaLoader(query=state['question'], 
                                  load_max_docs=2).load()

     # 格式化
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}">\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]} 

def generate_answer(state):
    
    """ 回答问题的节点 """

    # 获取状态
    context = state["context"]
    question = state["question"]

    # 模板
    answer_template = """Answer the question {question} using this context: {context}"""
    answer_instructions = answer_template.format(question=question, 
                                                       context=context)    
    
    # 回答
    answer = llm.invoke([SystemMessage(content=answer_instructions)]+[HumanMessage(content=f"Answer the question.")])
      
    # 追加到状态
    return {"answer": answer}

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("search_web",search_web)
builder.add_node("search_wikipedia", search_wikipedia)
builder.add_node("generate_answer", generate_answer)

# 流程
builder.add_edge(START, "search_wikipedia")
builder.add_edge(START, "search_web")
builder.add_edge("search_wikipedia", "generate_answer")
builder.add_edge("search_web", "generate_answer")
builder.add_edge("generate_answer", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"question": "How were Nvidia's Q2 2024 earnings"})
result['answer'].content

## 使用 LangGraph API

**⚠️ 免责声明**

自这些视频录制以来，我们已经更新了 Studio，使其可以在本地运行并在浏览器中打开。这现在是运行 Studio 的首选方式（而不是像视频中显示的使用桌面应用程序）。请参阅有关本地开发服务器的文档[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)。要启动本地开发服务器，请在此模块的 `/studio` 目录中在终端中运行以下命令：

```
langgraph dev
```

您应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到 Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [ ]:
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

In [ ]:
thread = await client.threads.create()
input_question = {"question": "How were Nvidia Q2 2024 earnings?"}
async for event in client.runs.stream(thread["thread_id"], 
                                      assistant_id="parallelization", 
                                      input=input_question, 
                                      stream_mode="values"):
    # 检查答案是否已添加到状态
    if event.data is not None:
        answer = event.data.get('answer', None)
        if answer:
            print(answer['content'])